In [12]:
!pip install -q transformers datasets

In [36]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

In [37]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [38]:
from google.colab import files
uploaded = files.upload()

In [39]:
train_df = pd.read_csv("/content/trainV2.csv")
test_df = pd.read_csv("/content/TestV2 - testV2.csv")

train_df.head()

,Text,Class
0,நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...,Non-Abusive
1,உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...,Abusive
2,கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...,Non-Abusive
3,ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...,Abusive
4,ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...,Non-Abusive


In [42]:
train_df['Class'].unique()

array(['Non-Abusive', 'Abusive', 'abusive'], dtype=object)

In [44]:
train_df.columns = ["text", "label"]
test_df.columns = ["text"]

In [45]:
label_map = {'Non-Abusive': 0, 'Abusive': 1, 'abusive':1}
train_df['label'] = train_df['label'].map(label_map)
print('Label distribution:\n', train_df['label'].value_counts())

Label distribution:
 label
0    1883
1    1769
Name: count, dtype: int64


In [46]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.1,
    random_state=42
)

In [47]:
train_dataset = Dataset.from_dict({
    "text": train_texts.tolist(),
    "label": train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    "text": val_texts.tolist(),
    "label": val_labels.tolist()
})

In [48]:
train_df.head()

,text,label
0,நான் கூட உன்னை வெகுளியான பொண்ணு&#39;னு நெனச்சி...,0
1,உன் போட்டோவை டாய்லெட்டுக்கு மாற்றினார்கள் அசிங...,1
2,கண்டா வரச்சொல்லுங்க கார்த்திய திவ்யாவோட சேர்த்...,0
3,ஒன்னோட சைசுக்கு நீயே ஒரு நாளக்கி 5கிலோ ஆய் போவ...,1
4,ரெண்டும் மிக பெரிய வெடிகுண்டு இவங்கள எதுக்கு ஷ...,0


In [49]:
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [50]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/3286 [00:00<?, ? examples/s]

Map:   0%|          | 0/366 [00:00<?, ? examples/s]

In [51]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expe

In [52]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    report_to="none"
)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [54]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.559916
2,No log,0.501210
3,0.556751,0.453929


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=618, training_loss=0.528020747656961, metrics={'train_runtime': 329.6443, 'train_samples_per_second': 29.905, 'train_steps_per_second': 1.875, 'total_flos': 648437195934720.0, 'train_loss': 0.528020747656961, 'epoch': 3.0})

In [55]:
test_dataset = Dataset.from_dict({"text": test_df["text"].tolist()})
test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset.set_format("torch", columns=["input_ids","attention_mask"])

Map:   0%|          | 0/913 [00:00<?, ? examples/s]

In [56]:
predictions = trainer.predict(test_dataset)
preds = torch.argmax(torch.tensor(predictions.predictions), dim=1).numpy()

In [57]:
output_df = pd.DataFrame({
    "text": test_df["text"],
    "label": preds   # 1 Non-Abusive, 0 Abusive
})

output_df.to_csv("muril_predictions.csv", index=False)

In [58]:
files.download("muril_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>